# HEATWIN — Phase 3 identification
Reproduce identified models from the tested Phase 2 simulator. Run from the repository root.

In [ ]:
from pathlib import Path
from heatwin.config import load_config
from heatwin.identification.step_test import create_step_experiment, measure_process_gain
from heatwin.identification.fit_models import fit_step_response
from heatwin.identification.linearization import linearize_at_valve, finite_difference_jacobians
from heatwin.identification.validation import error_metrics


In [ ]:
p = load_config()
training = create_step_experiment(p, u0=0.40, delta_u=0.10)
validation = create_step_experiment(p, u0=0.40, delta_u=0.05, split='validation')
print('Measured gain (°C per valve fraction):', measure_process_gain(training.data, 60))


In [ ]:
fits = {name: fit_step_response(training.data, model=name, step_time_s=60) for name in ('fo','fopdt','second_order')}
for name, fit in fits.items():
    prediction = fit.predict(validation.data.time_s.to_numpy(), delta_u=0.05, baseline_c=validation.equilibrium.tc)
    mask = validation.data.time_s >= 60
    print(name, fit.process_gain, fit.tau1_s, fit.theta_s, error_metrics(validation.data.tc_c.to_numpy()[mask], prediction[mask]).rmse)


In [ ]:
local = linearize_at_valve(p, 0.4)
A_num, B_num = finite_difference_jacobians(p, 0.4)
print('A=', local.A, '\nB=', local.B, '\nPoles=', local.poles)
print('max |A-A_fd|=', abs(local.A - A_num).max(), 'max |B-B_fd|=', abs(local.B - B_num).max())


In [ ]:
from scripts.phase3_identification import run
report = run(Path('outputs/phase3'))
print('Selected model:', report['selected_model']['selected_model'])
